# 05 - FISPACT activation studies

**~25 minutes.**

### The story

The transport run is done and signed off. Now the question changes: the machine has been
operated for years, it is shut down, and somebody has to go inside to replace a component.
**What is still radioactive, how much heat does it release, and which nuclides are to blame?**

That is an activation calculation, and it is FISPACT-II's job. F4Enix sits on both ends of it:

* **before** the run, it turns the MCNP results and the operating history into FISPACT inputs:
  the neutron spectrum, the collapse and condense runs, and the inventory input itself;
* **after** the run, it reads the output back into DataFrames: dose, decay heat, activity, and
  the reaction pathways that produced each nuclide.

FISPACT-II is not run in this workshop: the output you will read in Part B is already in
`data/fispact`.

### What you will use

`Spectra`, `GROUP_STRUCTURES`, `Pulse`, `IrradiationScenario`, `Material`, `FispactInp`,
`FispactOutput`, `PathwayCollection`.

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

In [ ]:
import numpy as np
import pandas as pd

from f4enix.core.constants import TIME_UNITS
from f4enix.core.egroups import GROUP_STRUCTURES
from f4enix.core.irradiation import IrradiationScenario, Pulse
from f4enix.core.spectra import Spectra
from f4enix.input.fispact_inp import FispactInp
from f4enix.input.MCNPinput import Input
from f4enix.output.fispact_legacy_out import FispactOutput, PathwayCollection
from f4enix.output.mctal import Mctal

FISP = DATA / "fispact"      # the FISPACT files used by this notebook
OUT_FISP = OUT / "fispact"   # everything this notebook writes
OUT_FISP.mkdir(exist_ok=True)

## 1. The activation workflow

An activation study is a chain, and FISPACT-II is only the middle of it:

```
              PART A - preparation                         PART B - post-processing

 mctal, tally 4 ──► Spectra ──► fluxes, collapse.i, condense.i
                                                \
 MCNP material ──┐                               ──► FISPACT-II ──► .out ──► FispactOutput
                 ├──► FispactInp ──► inventory input /                          │
 operating   ────┘                                                dose · heat · activity
 history (IrradiationScenario)                                         pathways
```

The **collapse** run folds the cross sections over your neutron spectrum, the **condense** run
prepares the decay data, and the **inventory** run irradiates and cools the material. F4Enix
writes the input of all three.

---

# Part A - Preparing the inputs

## 2. The neutron spectrum, from the mctal

FISPACT wants one flux value per energy group. Tally 4 of the ITER_1D run is exactly that:
a cell flux binned in energy. This is the same `Mctal` object as in notebook 03, and the
`Total` rows must go before anything else happens.

We look at **cell 53**: the tungsten armour on the outboard side of the plasma chamber,
made of material `M74`.

In [ ]:
mctal = Mctal(DATA / "iter_1d" / "mctal")
mctal.remove_totals(4)
flux = mctal.tallydata[4]

cell = 53
spectrum = flux[flux["Cells"] == cell]

print("energy groups:", len(spectrum))
spectrum.head()

175 groups is not a coincidence: the tally was set up on **VITAMIN-J-175**, one of the
group structures FISPACT understands. `GROUP_STRUCTURES` carries twenty of them, always in
**eV** and always as bin *boundaries*, so a 175-group structure has 176 numbers.

In [ ]:
ebins = GROUP_STRUCTURES["VITAMIN-J-175"]

print("some structures :", sorted(GROUP_STRUCTURES)[:6], "...")
print("boundaries      :", len(ebins), "-> groups:", len(ebins) - 1)
# the mctal reports the upper bound of each group, in MeV
print("same bins as t4 :", np.allclose(spectrum["Energy"] * 1e6, ebins[1:], rtol=1e-3))

`Spectra` holds the **shape** of the spectrum: values normalised to 1. The intensity is kept
apart, and it is worth keeping: tally 4 carries `FM4 1.222e21`, the source normalisation of the
run in n/s, so its sum is the total flux in the cell in n/cm²/s. That number comes back in
section 6.

> **Careful:** energies in eV, values normalised, and one value fewer than boundaries. Getting
> any of the three wrong raises a `ValueError` or, worse, silently shifts the whole spectrum.

In [ ]:
values = spectrum["Value"].to_numpy()
total_flux = values.sum()                      # n/cm2/s at full power
w_spectrum = Spectra(ebins, values / total_flux, name=f"ITER1D_C{cell}")

print(f"total flux : {total_flux:.3e} n/cm2/s")
print("name       :", w_spectrum.name)
print("sum        :", w_spectrum.spectra_values.sum())

## 3. Comparing with a reference spectrum

A FISPACT `fluxes` file contains only the numbers - no energies. So `from_fispact` needs you to
say which group structure they belong to. Get that wrong and the file still reads perfectly,
which is why the group structure is worth double-checking.

In [ ]:
reference = Spectra.from_fispact(GROUP_STRUCTURES["VITAMIN-J-175"], FISP / "FLUXESS1")

print("name written in the file:", reference.name)
print("groups                  :", len(reference.spectra_values))

`plot` draws one or several spectra together; `lethargy=True` plots per unit lethargy, which is
how spectra spanning ten decades of energy are usually compared.

In [ ]:
fig, ax = w_spectrum.plot(lethargy=True, add_spectra=[reference])

## 4. The collapse and condense inputs

`to_fispact_fluxes` writes the spectrum in the `fluxes` format the collapse reads (the file is
named after the spectrum), `to_arb_flux_file` writes it for a group conversion, and the two
`print_*` methods write the small collapse and condense runs.

The component is tungsten, so the collapse needs the **self-shielding** of the large W186(n,γ)
resonances: `tungsten=True` adds the `PROBTAB` / `SSFCHOOSE` / `SSFMASS` block for you. The
`SSFMASS` composition is a built-in ITER-grade tungsten, not `M74` itself - close enough for
the self-shielding factors, but worth knowing.

In [ ]:
w_spectrum.to_fispact_fluxes(OUT_FISP)
w_spectrum.to_arb_flux_file(OUT_FISP)
w_spectrum.print_collapse_inp(OUT_FISP, tungsten=True)
w_spectrum.print_condensed_inp(OUT_FISP)

print(sorted(p.name for p in OUT_FISP.iterdir()))
print()
print((OUT_FISP / "collapse.i").read_text())

`GETXS 1 175` is the line that ties the collapse to the group structure of the flux above.

## 5. The irradiation history

A `Pulse` is a length of time at a given intensity. A pulse with intensity `0` is a dwell, and
that is how you describe a pulsed machine. An `IrradiationScenario` is the ordered list of
pulses, plus the cooling times at which you want the results.

Here the intensity is written as a **fraction of full power**: 1 is a full-power pulse. This
keeps the scenario independent of where the component sits; the flux of the cell is applied
only when the input is written, in section 6.

In [ ]:
pulsed = IrradiationScenario(
    [Pulse(2, 0.05, TIME_UNITS.YEAR)]            # two years at 5% of full power
    + [Pulse(3290, 0), Pulse(400, 1.0)] * 17,    # then 17 full-power pulses of 400 s
    name="two_years_then_pulses",
)
pulsed.set_cooling_times([(1, TIME_UNITS.HOUR), (1, TIME_UNITS.DAY), (1, TIME_UNITS.YEAR)])

print("cooling labels:", pulsed.cooling_labels)
pulsed.get_collapsed_table()

`get_collapsed_table` folds repeated sequences into a single row with a multiplier, which is
the only way a scenario with a few hundred pulses is readable. Cooling times are given
**after shutdown** and are cumulative by default, and each gets a short label.

Most of the time you do not type the scenario at all: it already exists in a FISPACT input, or
in a legacy `d1stime` script. `inp_fispact.i` is the input of the FISPACT run you will read in
Part B.

In [ ]:
scenario = IrradiationScenario.from_fispact(FISP / "inp_fispact.i")

print("pulses        :", len(scenario.pulses))
print("cooling labels:", scenario.cooling_labels)
scenario.get_collapsed_table()

The cooling labels read from a FISPACT input are just cumulative seconds. Setting the same
times again with their units gives labels a human can read - and these are the labels Part B
will use.

In [ ]:
scenario.set_cooling_times([
    (1, TIME_UNITS.DAY), (11.6, TIME_UNITS.DAY), (30, TIME_UNITS.DAY),
    (180, TIME_UNITS.DAY), (1, TIME_UNITS.YEAR), (10, TIME_UNITS.YEAR),
])
print("cooling labels:", scenario.cooling_labels)

`to_ascii` writes the whole scenario to a readable `.irr` file that `from_ascii` reads back - a
much better thing to put under version control than a FISPACT input.

In [ ]:
pulsed.to_ascii(OUT_FISP / "pulsed.irr")
print((OUT_FISP / "pulsed.irr").read_text())

reloaded = IrradiationScenario.from_ascii(OUT_FISP / "pulsed.irr")
print("same pulses after the round trip:", reloaded.pulses == pulsed.pulses)

## 6. The inventory input

`FispactInp` is a light wrapper around `pypact.InputData` that understands F4Enix objects:
`add_material` takes a `Material`, `add_irradiation_scenario` takes an `IrradiationScenario`.

The material is `M74` from the MCNP input - the same one that fills cell 53.

In [ ]:
inp = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")
w_armour = inp.mat_section["M74"]

print("elements in M74:", len(w_armour.elements))

Two arguments deserve attention:

* `style="mass"` writes a `MASS` block with **mass fractions per element**. FISPACT then uses
  its own natural abundances, so the isotopic detail of the MCNP material is not carried over.
  `style="fuel"` writes every isotope explicitly instead (it needs a density as well);
* `norm` multiplies every pulse intensity. Our pulses are fractions of full power, so
  `norm=total_flux` turns them into the absolute flux FISPACT expects, in n/cm²/s.

In [ ]:
fisp_inp = FispactInp()
fisp_inp.add_material(w_armour, style="mass", mass=1.0)     # 1 kg of M74
fisp_inp.add_irradiation_scenario(pulsed, norm=total_flux)

The rest of the input is set on the underlying `pypact.InputData`, available as `fisp_inp.inp`.
The collapse and condense of section 4 run separately, so the inventory run only needs to
**read** their results: that is `GETXS 0` and `GETDECAY 0`.

In [ ]:
fisp_inp.inp.readXSData(0)                    # GETXS 0: read the collapsed cross sections
fisp_inp.inp.readDecayData(condense=False)    # GETDECAY 0: read the condensed decay data
fisp_inp.inp.enableHalflifeInOutput()         # HALF
fisp_inp.inp.enableHazardsInOutput()          # HAZARDS
fisp_inp.inp.setAtomsThreshold(1e5)           # MIND 1e5

w_inp_file = OUT_FISP / "w_armour.i"
fisp_inp.save(w_inp_file)
print(w_inp_file.read_text()[:1500])

### Two keywords pypact cannot write

`DOSE` (the contact dose rate) and `UNCERT` (the uncertainties **and the pathways**) are not
supported by pypact. Without them the output has no dose column and no `GENERIC PATHWAYS`
block, and half of Part B would have nothing to read. A plain text edit is enough: add them to
the initialisation phase.

In [ ]:
def add_dose_and_uncert(path: Path) -> str:
    """Add DOSE 1 and UNCERT 2 to an input written by FispactInp (only once)."""
    text = path.read_text()
    if "\nDOSE " not in text:
        text = text.replace(
            "<< INITIALIZATION PHASE >>",
            "<< INITIALIZATION PHASE >>\n"
            "<< contact dose rate, uncertainties and pathways >>\n"
            "DOSE 1\n"
            "UNCERT 2",
        )
        path.write_text(text)
    return text


text = add_dose_and_uncert(w_inp_file)

# the head of the input, comments stripped
print("\n".join([line for line in text.splitlines() if line and not line.startswith("<<")][:12]))

The input is a FISPACT input like any other, so `IrradiationScenario.from_fispact` can read it
back. Dividing by `total_flux` must give the original fractions of power: a cheap check that
the normalisation went where it should.

In [ ]:
check = IrradiationScenario.from_fispact(w_inp_file)

written = np.array([p.intensity for p in check.pulses]) / total_flux
original = np.array([p.intensity for p in pulsed.pulses])
print("pulses          :", len(check.pulses), "vs", len(pulsed.pulses))
print("same intensities:", np.allclose(written, original))

Together with `collapse.i`, `condense.i` and the `fluxes` file of section 4, this is a complete
set of FISPACT-II inputs.

---

# Part B - Reading the output

## 7. Opening a FISPACT output

A legacy FISPACT-II `.out` is a few tens of MB of fixed-format text holding every time step of
the inventory. `testSS.out` is the output of `inp_fispact.i`, the input `scenario` was read
from in section 5: a copper alloy, irradiated for about 14 years and then cooled.

`FispactOutput` needs the **labels of the cooling times**. It attaches them to the **last**
time steps of the file, in order, and checks nothing: give it the wrong number of labels and
they silently land on the wrong steps. The safest labels are the ones of the scenario that
produced the run.

In [ ]:
fispact_out = FispactOutput(FISP / "testSS.out", scenario.cooling_labels)

print("time steps in the file:", len(fispact_out.inventory_data))
print("cooling labels        :", fispact_out.cooling_labels)
print("cooling times [s]     :", [f"{t:.3e}" for t in fispact_out.cooling_times])

`cooling_times` is read from the file, the labels come from you: comparing the two is the check
that they agree. The tolerance is 1% because "11.6d" is itself a rounded label: the run cools for
exactly 10⁶ s.

In [ ]:
expected = np.cumsum([p.time for p in scenario.cooling_times])
print("labels match the file:", np.allclose(fispact_out.cooling_times, expected, rtol=1e-2))

FISPACT marks with a `?` the nuclides whose inventory did not converge. F4Enix checks them when
the file is opened, logs a warning for those that carry more than 1% of the activity, and lists
them in `problematic_isotopes`. An empty set is what you want to see.

In [ ]:
print("poorly converged:", fispact_out.problematic_isotopes or "none")

## 8. Dose, heat and activity

The three main results are plain DataFrames, one row per nuclide per cooling time, already
sorted by contribution. `isotope % dose` is the share of that nuclide, `Cumulative dose sum`
the running total - so the first rows are the ones that matter. They are parsed only the first
time you ask for them.

In [ ]:
fispact_out.sddr.head()

In [ ]:
fispact_out.decay_heat.head()

In [ ]:
fispact_out.activity.head()

Being DataFrames, the totals per cooling time are one `groupby` away.

In [ ]:
totals = pd.DataFrame({
    "dose [Sv/h]": fispact_out.sddr.groupby("cooling time", sort=False)["dose"].sum(),
    "heat [kW]": fispact_out.decay_heat.groupby("cooling time", sort=False)["heat"].sum(),
    "activity [Bq]": fispact_out.activity.groupby("cooling time", sort=False)["activity"].sum(),
})
totals

## 9. How it evolves in time

`plot_trend` follows, over all the cooling times, every nuclide needed to reach a given share
of a quantity: `"dose"`, `"heat"` or `"activity"`.

In [ ]:
fig, ax = fispact_out.plot_trend("dose", 95)

## 10. Who drives the dose

`filter_by_cum_dose` cuts the table where the cumulative contribution reaches the percentage you
ask for, at one cooling time. This is the answer to "which nuclides do I need to worry about".
`filter_by_cum_heating` and `filter_by_cum_activity` do the same for the other two quantities.

In [ ]:
fispact_out.filter_by_cum_dose(95, "11.6d")

Knowing *which* nuclide is only half the story - you also want to know **which reaction made
it**, because that is what you can act on in the material specification. `add_pathways=True`
explodes every nuclide row into one row per production pathway, and `pathway % dose` splits its
share among them.

In [ ]:
with_paths = fispact_out.filter_by_cum_dose(95, "11.6d", add_pathways=True)
with_paths[["element", "isotope", "state", "isotope % dose", "pathway", "pathway % dose"]]

## 11. The pathways themselves

The pathways come from the `GENERIC PATHWAYS` block of the output - the one `UNCERT 2` turns
on. They are available as `fispact_out.pathways_collection`, or on their own with
`PathwayCollection.from_file`, which does not need the rest of the output.

Each `Pathway` knows its parent, its intermediates and its reactions. `reduce()` strips the
isomeric transitions, which are not real steps, and `get_MT()` gives the ENDF reaction number
of the first reaction - the number a D1S `reac` file needs.

In [ ]:
pathways = fispact_out.pathways_collection
print("pathways found:", len(pathways.pathways))

path = pathways.pathways[13]
print("as written :", path)
print("multistep  :", path.is_multistep())
print("reduced    :", path.reduce())
print("first MT   :", path.get_MT())

In [ ]:
pathways.to_dataframe().head(10)

---

# Exercises

## Exercise 1 - an inventory input for the steel behind the armour

Cell 55 sits just behind the tungsten armour, and it is made of `M11`, the 316L(N)-IG
stainless steel. Prepare its activation run with the `pulsed` scenario of section 5.

1. extract the spectrum of cell 55 from `flux`, keep its total flux in `steel_flux` and build
   a normalised `Spectra` called `steel_spectrum`, named `"ITER1D_C55"`;
2. write its `fluxes` file to `OUT_FISP`;
3. build a `FispactInp` called `steel_inp` with 1 kg of `M11` in `"mass"` style and the
   `pulsed` scenario at the flux of cell 55;
4. make it read the collapsed cross sections and the condensed decay data (`GETXS 0`,
   `GETDECAY 0`), and save it to `steel_file = OUT_FISP / "steel_c55.i"`;
5. add `DOSE 1` and `UNCERT 2` to the saved file.

<details>
  <summary><b>Hints</b> <- Click to expand</summary>

* section 2 does steps 1 and 2 for cell 53: `flux[flux["Cells"] == 55]`, then
  `Spectra(ebins, values / values.sum(), name=...)` and `to_fispact_fluxes(OUT_FISP)`;
* the material is `inp.mat_section["M11"]`;
* `add_irradiation_scenario(pulsed, norm=steel_flux)` - the pulses are fractions of full power;
* `readXSData(0)` and `readDecayData(condense=False)` on `steel_inp.inp`;
* `add_dose_and_uncert` of section 6 works on any file saved by `FispactInp`.

</details>

In [ ]:
# TODO 1: the spectrum of cell 55 and its total flux
steel_flux = ...
steel_spectrum = ...

# TODO 2: the fluxes file


# TODO 3: the inventory input with 1 kg of M11 and the pulsed scenario
steel_inp = ...

# TODO 4: read the collapse and condense results, then save
steel_file = OUT_FISP / "steel_c55.i"


# TODO 5: add DOSE 1 and UNCERT 2


In [ ]:
# --- self-check, do not modify ---
expected_flux = flux[flux["Cells"] == 55]["Value"].sum()
assert abs(steel_flux / expected_flux - 1) < 1e-9, (
    f"steel_flux is {steel_flux:.4e}, the total flux of cell 55 is {expected_flux:.4e}"
)
assert steel_spectrum.name == "ITER1D_C55", f"the spectrum is named {steel_spectrum.name!r}"
assert abs(steel_spectrum.spectra_values.sum() - 1) < 1e-6, "the spectrum is not normalised"
reread = Spectra.from_fispact(ebins, OUT_FISP / "ITER1D_C55")
assert np.allclose(reread.spectra_values, steel_spectrum.spectra_values, rtol=1e-2), (
    "the fluxes file of cell 55 does not contain its spectrum"
)

text = steel_file.read_text()
for keyword in ("GETXS 0", "GETDECAY 0", "MASS 1.0", "DOSE 1", "UNCERT 2"):
    assert keyword in text, f"{keyword!r} is missing from {steel_file.name}"
iron = [float(line.split()[1]) for line in text.splitlines() if line.startswith("Fe ")]
assert iron and iron[0] > 50, "the MASS block should hold the steel: mostly iron"

written = IrradiationScenario.from_fispact(steel_file)
assert len(written.pulses) == len(pulsed.pulses), (
    f"{len(written.pulses)} pulses in the file, the pulsed scenario has {len(pulsed.pulses)}"
)
peak = max(p.intensity for p in written.pulses)
assert abs(peak / steel_flux - 1) < 1e-6, (
    f"the full-power pulses are written at {peak:.4e} n/cm2/s, expected {steel_flux:.4e}"
)
print("Exercise 1 correct")

## Exercise 2 - dose or heat, same culprits?

The shutdown dose rate is a shielding problem, the decay heat is a cooling problem. Both come
out of the same inventory, but not necessarily out of the same nuclides. Check it on
`fispact_out`, **one day** after shutdown.

1. get the nuclides that make up 95% of the **dose** and those that make up 95% of the **decay
   heat**, both at the `"1d"` cooling time, and call them `dose_drivers` and `heat_drivers`;
2. label the rows of the two tables with a `nuclide` column built from the element, the isotope
   and the state, then compute the two set differences `dose_only` and `heat_only` and print
   them together with the nuclides common to both lists;
3. build a table `comparison` with one row per nuclide of the **union** of the two lists,
   holding the `nuclide` label, its `isotope % dose` and its `isotope % heat`, plus a
   `heat/dose` column with the ratio of the two shares;
4. print the nuclide with the largest ratio. Why is it a cooling problem and not a shielding
   problem?

<details>
  <summary><b>Hints</b> <- Click to expand</summary>

* `filter_by_cum_dose(95, "1d")` and `filter_by_cum_heating(95, "1d")`; the label must be
  written exactly as it was given to `FispactOutput`;
* the state is `""` for a ground state and `"m"` for a metastable, so
  `df["element"] + df["isotope"].astype(str) + df["state"]` is a good label;
* the shares of **all** the nuclides, not only of the drivers, are in `fispact_out.sddr` and
  `fispact_out.decay_heat`: filter them on the cooling time, then keep your union of nuclides;
* `pd.merge(dose_part, heat_part, on="nuclide", how="outer")` puts the two share columns side
  by side.

</details>

In [ ]:
label = "1d"

# TODO 1: the two lists of drivers
dose_drivers = ...
heat_drivers = ...

# TODO 2: label both tables and compare the two sets of nuclides
dose_only = ...
heat_only = ...

# TODO 3: the shares of the union of the two sets, side by side
comparison = ...

# TODO 4: who has the largest heat/dose ratio?

comparison

In [ ]:
# --- self-check, do not modify ---
assert len(dose_drivers) == 5, f"the 95% dose table has {len(dose_drivers)} rows, expected 5"
assert len(heat_drivers) == 7, f"the 95% heat table has {len(heat_drivers)} rows, expected 7"
assert set(dose_only) == {"Ni57"}, f"the dose-only nuclide should be Ni57, you found {dose_only}"
assert set(heat_only) == {"Co57", "Cr51", "Mo99"}, (
    f"the heat-only nuclides should be Co57, Cr51 and Mo99, you found {heat_only}"
)
for column in ("nuclide", "isotope % dose", "isotope % heat", "heat/dose"):
    assert column in comparison.columns, (
        f"no {column!r} column in comparison, its columns are {list(comparison.columns)}"
    )
assert len(comparison) == 8, f"the comparison table has {len(comparison)} rows, expected 8"
worst = comparison.sort_values("heat/dose", ascending=False).iloc[0]
assert worst["nuclide"] == "Mo99", f"the largest heat/dose ratio is {worst['nuclide']}, not Mo99"
assert worst["heat/dose"] > 3.5, (
    f"Mo99 carries {worst['heat/dose']:.2f} times its dose share as heat, expected about 4"
)
print("Exercise 2 correct")

---

## What you can now do

| Question | Method |
|----------|--------|
| turn an MCNP spectrum into a FISPACT flux | `Spectra`, `GROUP_STRUCTURES`, `to_fispact_fluxes` |
| read a spectrum back from a `fluxes` file | `Spectra.from_fispact` |
| feed a group conversion instead | `Spectra.to_arb_flux_file` |
| compare two spectra | `Spectra.plot(lethargy=True, add_spectra=[...])` |
| write the collapse and condense inputs | `print_collapse_inp`, `print_condensed_inp` |
| self-shield a tungsten component | `print_collapse_inp(..., tungsten=True)` |
| describe an irradiation history | `Pulse`, `IrradiationScenario`, `set_cooling_times`, `get_collapsed_table` |
| reuse a scenario from an existing input | `IrradiationScenario.from_fispact`, `from_legacy_d1stime` |
| keep a scenario under version control | `to_ascii` / `from_ascii` |
| write the inventory input | `FispactInp.add_material`, `add_irradiation_scenario`, `.inp`, `save` |
| read a FISPACT output | `FispactOutput(file, cooling_labels)` |
| check the run | `.cooling_times`, `.problematic_isotopes` |
| get dose, heat and activity | `.sddr`, `.decay_heat`, `.activity`, `plot_trend` |
| find who drives a quantity | `filter_by_cum_dose`, `filter_by_cum_heating`, `filter_by_cum_activity` |
| get the reaction pathways | `add_pathways=True`, `PathwayCollection`, `Pathway.reduce`, `Pathway.get_MT` |

## Where this goes next

The pathways found here are the input of a **D1S-UNED** shutdown dose rate calculation, together
with the **time correction factors** of the irradiation scenario (`TCF_Computer`).
`ReactionFile.filter_by_paths` turns the pathways into a `reac` file and
`IrradiationFile.from_irradiation_schedules` turns the factors into an `irrad` file, and
`PathwayLibrary` ships the pathways of past F4E activation studies so that, for common
materials, you can skip the FISPACT run altogether. All of it is in the documentation:

* [Handling irradiation scenarios](https://f4enix.readthedocs.io/en/latest/examples/input/jupyters/irradiation.html)
* [Fispact Input](https://f4enix.readthedocs.io/en/latest/examples/input/jupyters/fispact_inp.html)
* [FISPACT II legacy output](https://f4enix.readthedocs.io/en/latest/examples/output/jupyters/fispact_legacy.html)
* [Setting up a D1SUNED study](https://f4enix.readthedocs.io/en/latest/tutorial/d1suned/d1suned_tutorial.html)